# Lab 6, part A: the repository and the routes

Lab 6 (A+B) budget: $0.30, or nothing beyond a Claude subscription

**Everything so far has called a model from a cell. This lab does not.**

Scenario 2, code generation with Claude Code. The project is a real repository, the client is
the `claude` CLI in a terminal, and the notebook's job is to build the working copy, put its
traps on screen before you meet them, and hold the prompts you type.

Part A is the repository, what it tells Claude about itself, and the three routes out of the
diagram: **direct execution, plan mode, and the Explore subagent.** Then the tests-first loop.

Part B, the same repository, is the session lifecycle: compaction, resume, fork, fresh.

## 1. The project, and your working copy

`mycorp/` sits next to this notebook and is checked into the course repository, so you can
read it before a session touches it. The refund path of the MyCorp online shop: thirteen
modules, four decision records, a thin test suite.

It ships without a `.git`, because a repository inside a repository is a problem for the
course checkout. The cell below makes your working copy and gives it a history: **two commits
and a `lab-start` tag**, so `git diff lab-start` shows everything you have done, at any point.

Set `REBUILD = True` to throw the copy away and start again. That is the reset.

In [ ]:
import labkit

lab = labkit.start(model_env=None, credential="none")

import mycorp_lab

REBUILD = False          # True throws the working copy away, losing anything in it

REPO, git = mycorp_lab.build(lab, rebuild=REBUILD)

## 2. What the project tells Claude about itself

**`CLAUDE.md` is the first thing a session in this directory reads, and it is guidance rather
than enforced configuration.**

Four of its conventions decide almost everything below:

| Convention | What it decides later |
|---|---|
| A validator returns a message and never raises | Whether the first fix coerces the string or rejects it |
| Money is carried as floats | The size of the migration, and why it is a plan and not an edit |
| Historic names are re-exported twice | Why a search for one name finds a third of the callers |
| The gateway is the only network boundary | Which functions can be tested without patching |

The decision records carry the rest. **One of them, ADR-0007, says in as many words that no
decision has been taken**, which is the whole reason that task goes to plan mode.

In [ ]:
mycorp_lab.project_docs(REPO)

## 3. The failure you are about to hand over

**A stack trace that names a file, a function and a line is the clearest signal there is that
a task wants direct execution:** there is nothing to discover and nothing to choose between.

TKT-0053. The portal-initiated refund path sends the amount as a JSON string, and the
validator compares it to a ceiling before it checks what it is.

Run it, then hand the trace over as it is rather than describing it.

In [ ]:
print(mycorp_lab.pytest_tail(REPO, "tests/test_refund_amount.py"))

## 4. Size the task before you choose the route

**Three tickets, three shapes.** The test is CALM: **c**omplex architecture, **a**lternative
approaches, **l**arge file count, **m**ulti-step exploration. Any one of them true means plan
mode.

| Ticket | Shape | Route |
|---|---|---|
| TKT-0053, the string amount | One file, a trace that names the line | Direct execution |
| TKT-0055, floats to `Decimal` | Every module that carries an amount | Plan mode |
| TKT-0054, tell the customer | Two defensible designs, no decision taken | Plan mode |

The cell below is the measurement, not a guess: **Glob for the module paths, Grep each for an
amount, and count.** That is also the order a session should work in, per task 2.5: find the
entry points by content first, then read only what you have to. `CLAUDE.md` asks for exactly
that, and `mycorp_lab.money_modules` obeys it, so you can check a session's answer against a
number.

In [ ]:
for bucket, modules in mycorp_lab.money_modules(REPO).items():
    print(f"{bucket} ({len(modules)})")
    for relative in modules:
        print(f"  {relative}")

print("\nADR-0007, What is undecided:")
print(mycorp_lab.undecided(REPO))

## 5. Two traps, planted on purpose

**The alias chain.** `charge_card` is re-exported as `process_payment` in
`shop/billing/__init__.py`, and that name is re-exported again as `take_payment` in
`shop/legacy.py`. One caller still uses the original. So a search for any single name finds
about a third of the answer, and the way out is the one task 2.5 names: **list the exported
names first, then search for each.**

**The duplicate anchor.** `timeout_seconds = 5.0` appears twice in `shop/payments/gateway.py`,
identically, in two different methods. An `Edit` anchored on that line matches twice and
fails. The recovery is a longer anchor or a Read followed by a Write. **What must not happen
is `replace_all`**, which changes both.

In [ ]:
exports, callers = mycorp_lab.alias_chain(REPO)
for relative, names in exports:
    print(f"{relative:32} exports {names}")
print("the charging chain: charge_card then process_payment then take_payment\n")

print(f"{len(callers)} call sites, found only because every name in the chain was searched:")
for where, name in callers:
    print(f"  {where:26} {name}")
print("a search for any one name alone finds one of the three.\n")

anchor = "timeout_seconds = 5.0"
found = mycorp_lab.duplicate_anchor(REPO, "shop/payments/gateway.py", anchor)
print(f"'{anchor}' appears {len(found)} times in shop/payments/gateway.py:")
for number, line in found:
    print(f"  line {number}: {line}")

## 6. Three routes, live in Claude Code

Open a terminal beside this notebook and start the session in the working copy:

```bash
cd workspace/mycorp
claude
```

Everything below is one session. **Write the route down before each task: deciding afterwards
is not deciding.**

### One. Direct execution, because the trace names the line

> `tests/test_refund_amount.py` fails with `TypeError: '>' not supported between instances of
> 'str' and 'float'`, raised inside `RefundAmountValidator.check` in
> `shop/validators/refund_amount.py`. The portal-initiated refund path sends `refund_amount`
> as a JSON string, for example `"12.50"`. Read `tests/test_refund_amount.py` and
> `shop/validators/__init__.py` first, then fix the validator so all three tests pass. Do not
> coerce the string to a number: the pattern in this project says a validator returns a
> message for a value it will not accept, and never raises. One file. No plan.

CALM has nothing true here, which is what makes it direct execution. The judgement in it is
the one the ticket hides: **coercing would also make the test pass**, and `CLAUDE.md` is what
says it is wrong.

### Two. Plan mode, because the blast radius is the question

Shift+Tab cycles into plan mode, or start the session again with `claude --permission-mode plan`.

> Read `docs/adr/ADR-0003-money-as-floats.md`, then plan TKT-0055: move every amount in this
> project from `float` to `Decimal`. Plan only, no edits. List the affected modules one path
> per line, in the order you would change them, and for each one say what changes and what
> breaks if it is done out of order. Include `shop/payments/gateway.py`, where the wire
> format stays a JSON number, and say exactly where the conversion sits.

Section 4 counted the modules that carry an amount. **A plan that lists every file in the
repository has not answered the question.**

### Three. Plan mode again, because nobody has decided yet

> Read `docs/adr/ADR-0007-refund-notifications.md`. It says plainly that no decision has been
> taken. Plan TKT-0054 both ways: the refunds service sending the notification itself on the
> way out of `send_refund`, and writing a row to the existing `outbox` for
> `notifications-worker` to drain. For each design, give me the failure mode when the
> notification path is down, and name the constraint in the ADR that decides it. Recommend
> one and say why. Still no edits.

If it comes back with a single design:

> Give me the other one as well, with the same failure analysis, then choose.

**The reason it gives is the deliverable, not the design.**

### Four. Explore, because discovery is verbose and read-only

> Use the Explore subagent for this. Find every place in this repository where a card is
> actually charged, including calls that arrive under a re-exported name.
> `shop/billing/__init__.py` and `shop/legacy.py` re-export the card functions, and one
> caller still uses the original name, so list the exported names first and then search for
> each one. Return one line per call site, `path:line`, and nothing else.

Section 5 printed the answer, so you can check it. **Explore reads in its own context and
returns a summary**, so the greps never land in your window. It also skips `CLAUDE.md`, which
is why the alias rule is in the message rather than left to the project file.

### Five. A named subagent, for one specific question

> Use the `trace-gateway` subagent: which functions under `shop/` end up calling
> `shop/payments/gateway.py`, and for each one, what would a test have to supply instead of a
> real gateway? Say whether the current signature allows that without patching.

**Keep the answer. Section 8 needs it**: `send_refund`, `send_reversal` and `convert` are all
handed what they talk to, while `charge_card` and `refund_card` build their own client when
none is passed, which is what makes `shop/checkout.py` and `shop/jobs.py` untestable.

### Six. The edit that matches twice

> In `shop/payments/gateway.py`, raise the timeout in the `refund` method from 5.0 to 12.0
> and leave `charge` alone.

Watch the recovery when the `Edit` reports two matches: a longer anchor that takes the
`def refund(` line with it, or a Read and a Write. Then check, and commit the fix:

```bash
git diff -- shop/payments/gateway.py
git add -A && git commit -m "TKT-0053: return a message for a non-numeric refund amount"
```

One changed line in the gateway, and a validator that rejects rather than raises.

## 7. The tests-first loop, before you run it

**The order is the one thing about this work that cannot be reconstructed afterwards**, which
is why it goes in the history rather than in a note: tests written and committed while they
still fail, then the implementation committed separately.

Two tickets feed it, and each one is a different reason prose was not enough:

- **TKT-0051**, `normalise_reason`, raises `NotImplementedError`. The ticket describes the
  rules in prose and the prose has already produced two different readings. **Examples are the
  answer**, and one of them has to be the case the prose keeps losing.
- **TKT-0052**, the rate cache. Nobody has decided what happens when the gateway is
  unreachable and the cached rate has expired. Asking for code first would invent an answer,
  so **the session interviews you instead.**

The cell below prints what is there now, and computes the gap the session is about to map.

In [ ]:
print((REPO / "shop/refunds/reasons.py").read_text(encoding="utf-8"))

rates = (REPO / "shop/rates.py").read_text(encoding="utf-8")
print("shop/rates.py defines:")
for line in rates.splitlines():
    if line.startswith("def "):
        print(f"  {line}")

print("\nmodules under shop/ that no test imports:")
for relative in mycorp_lab.untested_modules(REPO):
    print(f"  {relative}")

## 8. Tests first, live in Claude Code

Same terminal, same session if it is still open.

### One. Map before you plan

> Which files in this project are tests, and which module does each one cover? Use Glob for
> the paths. Then list the modules under `shop/` that no test touches at all. Do not read
> every file: grep for the imports and read only what you have to.

### Two. The prioritised plan, and the dependency that reorders it

`/gaps` is a project slash command, in `.claude/commands/gaps.md`, so it is only offered in a
session started here.

```
/gaps the whole shop
```

Then let it see the seam:

> Now read `shop/billing/cards.py` and `shop/payments/gateway.py`, and tell me which items on
> that list cannot be tested as the code stands today. Re-order the plan with that in it, and
> say what moved and why.

**The plan changing is the exercise, not a fault in the first plan.** "Stand the gateway in
first" is the shape of the answer.

### Three. Examples, because the prose has already failed twice

> `shop/refunds/reasons.py` has `normalise_reason` raising `NotImplementedError`. Here is
> what it has to do, as examples rather than prose:
>
>     "CUST_DAMAGED"        -> "damaged"
>     "other: cracked lid"  -> "other"
>     None                  -> "unknown"
>
> `other` is a reason the customer gave. `unknown` is the absence of one: the gateway omits
> the field entirely on a merchant-initiated refund. Write the test cases first, one per
> example plus the empty string and an unrecognised code. No implementation yet.

That last line is the one the prose keeps losing, and it is the edge case task 3.5 asks for by
name.

### Four. Interview, because nobody has decided this either

> `shop/rates.py` asks the gateway for a rate on every conversion, so a page showing twenty
> prices makes twenty calls. TKT-0052 wants a cache in front of it. Do not write any code
> yet. Ask me the questions you need answered before you could write it. Questions only.

Answer them. One will be what happens when the gateway is unreachable and the cached rate has
expired, which is the decision nobody had taken. Then pin the two things the tests depend on:

> Two things are fixed: `convert` keeps its signature, including the `source` argument, and
> the module exposes `clear_rate_cache()` so a test can start from a known state.

### Five. The tests, red, and the commit that proves the order

> Write the tests now, and only the tests. `tests/test_reasons.py` for the five reason cases.
> `tests/test_rates_cache.py` for three things: a second conversion of the same pair that does
> not reach the source, an entry that is refetched after the TTL, and `clear_rate_cache()`
> putting it back to empty. Use a fake source object that counts calls. Nothing in a test may
> touch the network. No implementation.

```bash
python -m pytest -q
git add tests/ && git commit -m "TKT-0051, TKT-0052: tests before the implementation"
```

**Red, and that is the point.** The commit goes in while they are still failing.

### Six. Implement, by sharing the failures

> Here is the failure output from `pytest -q`. Implement `normalise_reason` and the cache in
> `shop/rates.py` so these pass, and change nothing else.
>
>     [paste the output]

```bash
python -m pytest -q
git add -A && git commit -m "TKT-0051, TKT-0052: implement the reason codes and the rate cache"
```

## 9. What the commits prove

**The history is the evidence, and it is the part a summary cannot reconstruct.** Two commits
for the work, in that order, with the tests failing at the first one.

In [ ]:
print(git("log", "--oneline"))
print()
print(git("diff", "--stat", "lab-start") or "nothing changed since lab-start yet")
print()
print(mycorp_lab.pytest_tail(REPO, lines=1))

## What part A settled

**Three routes, chosen by the shape of the task rather than by habit:** a trace that names a
line goes direct, a blast radius goes to plan mode, and a decision nobody has taken goes to
plan mode twice. Discovery that would flood a window goes to a subagent, which returns a
summary and never shows you its greps.

Then the loop underneath all of it: examples where prose failed, an interview where nobody
had decided, tests committed while they were still red, and the implementation committed
after.

Stop here. Part B is the same repository and one long session.